In [ ]:
# Bootstrap this notebook even when it is opened in a fresh Colab kernel.
from pathlib import Path
import urllib.request
_BOOTSTRAP_REPO = Path('/content/kltn')
_BOOTSTRAP_SCRIPT = _BOOTSTRAP_REPO / 'scripts' / 'colab_bootstrap.py'
if not _BOOTSTRAP_SCRIPT.exists():
    raw = 'https://raw.githubusercontent.com/maiphuowng205/kltn/31b1b9f/scripts/colab_bootstrap.py'
    urllib.request.urlretrieve(raw, '/content/colab_bootstrap.py')
    _BOOTSTRAP_SCRIPT = Path('/content/colab_bootstrap.py')
exec(_BOOTSTRAP_SCRIPT.read_text(encoding='utf-8'), globals())


# M3–M5 risk, optimizer and benchmark backtest
Run Ledoit–Wolf risk estimation and the deterministic EW/EW-BH/MinVar/HM-MVO/Ridge-MVO/XGB-MVO/XGB-CA-MVO ladder.

In [ ]:
import subprocess, sys, json
baseline_run = WORKSPACE / 'runs' / 'v3_forecast_baselines'
portfolio_run = WORKSPACE / 'runs' / 'v3_portfolio_benchmarks'
subprocess.run([sys.executable, str(REPO / 'scripts' / 'run_v3_risk_coverage.py'), '--data-root', str(DATA_ROOT), '--run-dir', str(WORKSPACE / 'runs' / 'v3_risk_coverage')], check=True)
subprocess.run([sys.executable, str(REPO / 'scripts' / 'run_v3_portfolio_benchmarks.py'), '--data-root', str(DATA_ROOT), '--forecast-run', str(baseline_run), '--run-dir', str(portfolio_run)], check=True)
baseline_repeat = WORKSPACE / 'runs' / 'v3_forecast_baselines_repeat'
portfolio_repeat = WORKSPACE / 'runs' / 'v3_portfolio_benchmarks_repeat'
subprocess.run([sys.executable, str(REPO / 'scripts' / 'run_v3_forecast_baselines.py'), '--data-root', str(DATA_ROOT), '--run-dir', str(baseline_repeat)], check=True)
subprocess.run([sys.executable, str(REPO / 'scripts' / 'run_v3_portfolio_benchmarks.py'), '--data-root', str(DATA_ROOT), '--forecast-run', str(baseline_repeat), '--run-dir', str(portfolio_repeat)], check=True)
subprocess.run([sys.executable, str(REPO / 'scripts' / 'validate_v3_determinism.py'), '--main-run', str(baseline_run), '--repeat-run', str(baseline_repeat), '--files', 'forecasts.parquet', 'forecast_metrics_by_date.parquet', 'forecast_metrics_summary.parquet', '--report', str(WORKSPACE / 'runs' / 'v3_determinism_forecast.json')], check=True)
subprocess.run([sys.executable, str(REPO / 'scripts' / 'validate_v3_determinism.py'), '--main-run', str(portfolio_run), '--repeat-run', str(portfolio_repeat), '--files', 'portfolio_returns.parquet', 'weights.parquet', 'trades.parquet', 'solver_log.parquet', 'portfolio_metrics_summary.parquet', '--report', str(WORKSPACE / 'runs' / 'v3_determinism_portfolio.json')], check=True)
det = {'forecast': json.loads((WORKSPACE / 'runs' / 'v3_determinism_forecast.json').read_text()), 'portfolio': json.loads((WORKSPACE / 'runs' / 'v3_determinism_portfolio.json').read_text())}
(WORKSPACE / 'runs' / 'v3_determinism_report.json').write_text(json.dumps(det, indent=2))
import pandas as pd
pd.read_parquet(portfolio_run / 'portfolio_metrics_summary.parquet')